### Time-dependent Problem

Here we extend the Laplace problem from the first example by a time variable. This leads to the heat equation:

\begin{align*}
    \partial_t u -0.1\Delta u &= 1.0 &&\text{ in } (0, 2) \times \Omega \\
    u &= 0 , &&\text{ on } (0, 2) \times \partial \Omega \\
    u(0, \cdot) &= 0 &&\text{ in } \Omega
\end{align*}

with $\Omega = (0, 1) \times (0, 1)$.

In [22]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

Next we implement the *Variables* that appear in the problem:

In [23]:
import qewton
backend = qewton.DEFAULT_DL_BACKEND

X = qewton.Variable("x", dim=2)
U = qewton.Variable("u", dim=1)
T = qewton.Variable("t", 1) # <- Add the time variable "t" of dimension 1

Now we define our domain. The space domain is already completed, here you have to create the time interval and the Cartesian product of both.

In [24]:
omega = qewton.geometries.Rectangle(X, [0.0, 0.0], 1.0, 1.0)
time_interval = qewton.geometries.Interval(T, 0, 2.0) # <-add the bounds of the Interval (0, 2)
product_domain = time_interval * omega # TODO: Create the product domain of time and space. Products are define with: *

Next we need to create some points, this is done by the *Sampler*. Here we need 3, one inside the domain, one for the boundary and one for the initial condition.

In [25]:
# Add the product domain of time and space:
inner_sampler = qewton.RandomUniformSampler(product_domain, n_points=25000) 

# The boundary sampler is done already.
boundary_sampler = qewton.RandomUniformSampler(
    time_interval*omega.boundary, n_points=10000
)

# Currently only the left interval side {0} is passed in the initial sampler, 
# create the product domain with omega:
initial_sampler = qewton.RandomUniformSampler(
    time_interval.boundary_left*omega, n_points=5000
)

The neural network that learns the solution gets the time and space variable as an input and outputs the solution u. Add the correct spaces.

**Hint**: A product space is also defined with *

In [26]:
model = qewton.FCN(in_neurons=T*X, 
                   out_neurons=U, 
                   hidden_neurons=32, 
                   n_hidden_layers=3)

Now, we have to transform our mathematical conditions given by our PDE into corresponding training conditions.

First we handle the PDE itself. Here, you have to finish the residual function.

In [27]:
def pde_residual(u : U, x : X, t : T):
    # TODO: Pass in the correct variables for the derivative computation 
    # as the second argument below:
    return u.gradient(t) - 0.1*u.laplacian(x) - 1.0

pde_graph = qewton.PINNPipeline(
    sampler=inner_sampler,
    residual=pde_residual,
    models=[model]
)
pde_constraint = pde_graph.constraint

The boundary condition is already done, since it is the same as in the first example.

In [28]:
def boundary_residual(u : U):
    return u

boundary_constraint = qewton.constraints.PINNConstraint(
    boundary_residual, name="BoundaryConstraint", weight=50.0
)
boundary_graph = qewton.PINNPipeline(boundary_sampler, [model], boundary_constraint)

We now also need the initial condition. 

In [29]:
# Implement the residual for the initial condition:
def boundary_residual(u : U):
    return u

initial_graph = qewton.PINNPipeline(
    sampler=initial_sampler,
    residual=boundary_residual,
    models=[model]
)
initial_constraint = initial_graph.constraint

Start the training:

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=5000,
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=adam_phase,
    graphs=[pde_graph, boundary_graph, initial_graph],
    training_objectives=[pde_constraint, boundary_constraint, initial_constraint],
    device=qewton.cuda(0),
)

trainer.run()

Optimization Phase 1:  40%|████      | 2/5 [00:00<00:00, 56.74it/s, loss=3.13]

Optimization Phase 1: 100%|██████████| 5/5 [00:00<00:00, 70.83it/s, loss=1.62]


We can plot the solution:

In [33]:
layout = pde_graph.visualize(
    model.output_ports[0],
    controls={T: qewton.viz.TimeSpec(T, duration=150)}
)
qewton.viz.Figure(layout).show()

NotImplementedError: 